# Chat with Your SFT-Trained GPT Model

This notebook allows you to interact with your fine-tuned GPT model in a conversational format.
The model has been trained using Supervised Fine-Tuning (SFT) to respond appropriately to user messages.

## Instructions

1. **Load your trained model**: Update the model path in the first cell to point to your SFT-trained model
2. **Start chatting**: Use the chat interface to have conversations with your model
3. **Experiment**: Try different conversation styles, questions, and topics

## Features

- **Single-turn conversations**: Ask questions and get responses
- **Multi-turn conversations**: Maintain context across multiple exchanges
- **Adjustable parameters**: Control temperature and response length
- **Conversation history**: View the full conversation context

Enjoy chatting with your AI assistant! 🤖


In [ ]:
from IPython.display import HTML, display
from IPython import get_ipython

def set_css():
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap;
        }
    </style>
    '''))

ip = get_ipython()
if ip is not None:
    ip.events.register('pre_run_cell', set_css)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import os

src = "/content/drive/MyDrive/sft_model/sft_epoch_1.pth"
dst = "/content/drive/MyDrive/sft_model/sft_checkpoint_epoch_1.pth"

print("Loading:", src)
ckpt = torch.load(src, map_location="cpu")

if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    print("Found 'model_state_dict' in checkpoint, unwrapping it.")
else:
    state_dict = ckpt
    print("No 'model_state_dict' key, treating checkpoint as raw state_dict.")

unwanted_prefix = "_orig_mod."
fixed_any = False
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
        fixed_any = True
if fixed_any:
    print("Stripped '_orig_mod.' prefixes from state_dict keys.")

if any(k.startswith("embbeding.") for k in state_dict.keys()):
    print("Found 'embbeding.' typo in keys, fixing to 'embedding.'...")
    corrected = {}
    for k, v in state_dict.items():
        corrected[k.replace("embbeding.", "embedding.")] = v
    state_dict = corrected

torch.save(state_dict, dst)
print("Saved clean state_dict to:", dst)
print("File exists?", os.path.exists(dst))


Loading: /content/drive/MyDrive/sft_model/sft_epoch_1.pth
No 'model_state_dict' key, treating checkpoint as raw state_dict.
Stripped '_orig_mod.' prefixes from state_dict keys.
Saved clean state_dict to: /content/drive/MyDrive/sft_model/sft_checkpoint_epoch_1.pth
File exists? True


In [ ]:
# Import necessary libraries
import torch
import sft
import gpt
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


In [ ]:
# Configuration - UPDATE THESE PATHS FOR YOUR MODEL

# Path to your SFT-trained model checkpoint
MODEL_PATH = "/content/drive/MyDrive/sft_checkpoint_epoch_1.pth"

# Model configuration (should match your training configuration)
MODEL_CONFIG = {
    "vocab_size": 50262,
    "context_length": 1024,
    "emb_dim": 512,
    "n_heads": 8,
    "n_layers": 12,
    "drop_rate": 0.1,
}

# Generation parameters
MAX_NEW_TOKENS = 150
TEMPERATURE = 0.7

print(f"📁 Model path: {MODEL_PATH}")
print(f"⚙️  Model config: {MODEL_CONFIG}")
print(f"🎛️  Max tokens: {MAX_NEW_TOKENS}, Temperature: {TEMPERATURE}")


📁 Model path: /content/drive/MyDrive/sft_checkpoint_epoch_1.pth
⚙️  Model config: {'vocab_size': 50262, 'context_length': 1024, 'emb_dim': 512, 'n_heads': 8, 'n_layers': 12, 'drop_rate': 0.1}
🎛️  Max tokens: 150, Temperature: 0.7


In [ ]:
# Load tokenizer and model
print("🔄 Loading tokenizer...")
tokenizer = gpt.setup_tokenizer()
print(f"✅ Tokenizer loaded! Vocab size: {tokenizer.vocab_size}")

print("🔄 Loading model...")
model = sft.load_pretrained_model(MODEL_PATH, MODEL_CONFIG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"✅ Model loaded and moved to {device}!")


🔄 Loading tokenizer...
✅ Tokenizer loaded! Vocab size: 50257
🔄 Loading model...
Loading pre-trained model from /content/drive/MyDrive/sft_checkpoint_epoch_1.pth...
✅ Model loaded successfully!
✅ Model vocabulary size: 50262
✅ Model loaded and moved to cuda!


## Single-Turn Chat Interface

Use the function below to have individual conversations with your model. Each call is independent.


In [ ]:
def chat_once(user_message, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """
    Have a single conversation turn with the model.

    Args:
        user_message: Your message to the model
        max_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature (higher = more creative)

    Returns:
        Model's response
    """
    print(f"👤 You: {user_message}")
    print("🤖 Assistant: ", end="")

    response = sft.generate_chat_response(
        model=model,
        tokenizer=tokenizer,
        user_message=user_message,
        max_new_tokens=max_tokens,
        temperature=temperature
    )

    print(response)
    print("\n" + "="*50 + "\n")
    return response

# Test the chat function
print("🧪 Testing chat function...")
test_response = chat_once("Hello! How are you today?")
print("✅ Chat function working!")


🧪 Testing chat function...
👤 You: Hello! How are you today?
🤖 Assistant: What a fascinating question. One of the most significant challenges facing the human condition is the need for a hospital stay. The need for a hospital stay is often the result of a combination of factors, including the availability of medical care, the availability of affordable medical treatment, and the potential for complications and complications.

In addition, the remains of a hospital stay is crucial for ensuring that everyone is well-rested and prepared for the challenges of deceased individuals. With the assistance of a skilled medical professional, it's essential to ensure that everyone is on the same page, and that everyone is on the same page.

Additionally, the need for a hospital stay is also crucial. hospitals must prioritize staffing, staffing, and staffing for the necessary services


✅ Chat function working!


## Multi-Turn Chat Interface

Use the class below to maintain conversation context across multiple turns.


In [ ]:
class ChatSession:
    """
    A chat session that maintains conversation history.
    """

    def __init__(self, model, tokenizer, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
        self.model = model
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.conversation_history = []

    def add_message(self, role, content):
        """Add a message to the conversation history."""
        self.conversation_history.append({"role": role, "content": content})

    def chat(self, user_message):
        """
        Send a message and get a response, maintaining conversation context.

        Args:
            user_message: Your message to the model

        Returns:
            Model's response
        """
        # Add user message to history
        self.add_message("user", user_message)

        print(f"👤 You: {user_message}")
        print("🤖 Assistant: ", end="")

        # Generate response using conversation history
        response = sft.generate_multi_turn_response(
            model=self.model,
            tokenizer=self.tokenizer,
            conversation_history=self.conversation_history[:-1],  # Exclude the current user message
            max_new_tokens=self.max_tokens,
            temperature=self.temperature
        )

        # Add assistant response to history
        self.add_message("assistant", response)

        print(response)
        print("\n" + "="*50 + "\n")
        return response

    def clear_history(self):
        """Clear the conversation history."""
        self.conversation_history = []
        print("🗑️  Conversation history cleared!")

    def show_history(self):
        """Display the full conversation history."""
        print("📜 Conversation History:")
        print("="*50)
        for i, msg in enumerate(self.conversation_history):
            role_emoji = "👤" if msg["role"] == "user" else "🤖"
            print(f"{role_emoji} {msg['role'].title()}: {msg['content']}")
            if i < len(self.conversation_history) - 1:
                print("\n")
        print("="*50)

    def set_parameters(self, max_tokens=None, temperature=None):
        """Update generation parameters."""
        if max_tokens is not None:
            self.max_tokens = max_tokens
        if temperature is not None:
            self.temperature = temperature
        print(f"⚙️  Updated parameters: max_tokens={self.max_tokens}, temperature={self.temperature}")

# Create a chat session
chat_session = ChatSession(model, tokenizer)
print("✅ Chat session created!")


✅ Chat session created!


## Interactive Chat Examples

Try these example conversations to test your model!


In [ ]:
# Example 1: Single-turn conversation
print("🎯 Example 1: Single-turn conversation")
chat_once("What is machine learning?")


🎯 Example 1: Single-turn conversation
👤 You: What is machine learning?
🤖 Assistant: Machine learning is a crucial process of learning and developing new skills in various fields, from automation to software development. When an AI learns with minimal input and can learn from its experience, it's essential to develop a structured learning environment that is structured and tailored to the specific needs of the target audience.

Machine learning can be used to learn through hands-on experience, learning through hands-on experiences, and training in real-world scenarios. It's essential to develop a strong network of learning experiences, knowledge, and decision-making systems, which can be taught through a variety of channels.

Machine learning is an essential part of the learning process, as it can be used to learn from failures, improve the overall learning experience, and enhance




"Machine learning is a crucial process of learning and developing new skills in various fields, from automation to software development. When an AI learns with minimal input and can learn from its experience, it's essential to develop a structured learning environment that is structured and tailored to the specific needs of the target audience.\n\nMachine learning can be used to learn through hands-on experience, learning through hands-on experiences, and training in real-world scenarios. It's essential to develop a strong network of learning experiences, knowledge, and decision-making systems, which can be taught through a variety of channels.\n\nMachine learning is an essential part of the learning process, as it can be used to learn from failures, improve the overall learning experience, and enhance"

In [ ]:
# Example 2: Multi-turn conversation
print("🎯 Example 2: Multi-turn conversation")

# Start a new conversation
chat_session.clear_history()

# First turn
chat_session.chat("Hi! I'm learning about AI. Can you help me?")

# Second turn (maintains context)
chat_session.chat("What's the difference between supervised and unsupervised learning?")

# Third turn (continues the conversation)
chat_session.chat("Can you give me an example of each?")

# Show the full conversation history
chat_session.show_history()


🎯 Example 2: Multi-turn conversation
🗑️  Conversation history cleared!
👤 You: Hi! I'm learning about AI. Can you help me?
🤖 Assistant: Hello! How can I help you today? Well, let's start by understanding the specific topics you're interested in and what topics are most helpful for you. Here are some general questions to help you make a list of the topics you're interested in discussing:

1. **What topics do you enjoy, and what are your favorite foods?
2. **What topics do you enjoy and would like to discuss?
3. **What topics do you enjoy and what topics do you enjoy?
4. **What activities do you enjoy doing so much?
5. **What are your interests in the topic?
6. **What are some popular topics that you enjoy and what topics do you enjoy?
7. **What


👤 You: What's the difference between supervised and unsupervised learning?
🤖 Assistant: Sure, here are some topics that you can try to make a list of topics:

1. **What are some favorite foods and activities you enjoy?
2. **What are some popular

In [ ]:
# Example 3: Creative conversation
print("🎯 Example 3: Creative conversation")

# Adjust parameters for more creative responses
chat_session.set_parameters(max_tokens=200, temperature=0.9)

# Start a creative conversation
chat_session.clear_history()
chat_session.chat("Write a short story about a robot learning to paint.")
chat_session.chat("What happens next in the story?")

# Show the creative conversation
chat_session.show_history()


🎯 Example 3: Creative conversation
⚙️  Updated parameters: max_tokens=200, temperature=0.9
🗑️  Conversation history cleared!
👤 You: Write a short story about a robot learning to paint.
🤖 Assistant: Imagine your group is talking about a challenging situation. Make sure to:

* Have a friend explore the city's heart and mind: Was there time to visit, or was the people feeling anxious about how busy and seriously she's feeling?
* Have friends with her friends and family slowly transition away from your group or do they feel uncomfortable interrupting them?
* Have a partner listen to her colleagues, and then feel comfortable exchanging information
* Have a friend explore their favorite places and displays to create a more positive atmosphere?

Ultimately, the goal is to find a way to lead the conversation without feeling overwhelmed or stressed. By understanding the challenges ahead, you can create a loyal and supportive community that works both personally and in a plan.

For the chat, try

In [ ]:
# Adjust parameters for different response styles
print("🎛️  Adjusting parameters...")

# More conservative responses
chat_session.set_parameters(max_tokens=100, temperature=0.3)
chat_session.chat("Explain quantum computing simply.")

# More creative responses
chat_session.set_parameters(max_tokens=200, temperature=1.0)
chat_session.chat("Write a haiku about programming.")


🎛️  Adjusting parameters...
⚙️  Updated parameters: max_tokens=100, temperature=0.3
👤 You: Explain quantum computing simply.
🤖 Assistant: Your friend's concerns about the situation are a bit vague, but they can be a bit vague and may not be immediately apparent from the conversation. Here's a suggested response:

"I want to ask if a situation is going wrong, but I'm not sure what to do. I want to know if a situation is going wrong, and I want to know if there's anything I can do to help resolve it."

By asking, "What can I do to help you?"


⚙️  Updated parameters: max_tokens=200, temperature=1.0
👤 You: Write a haiku about programming.
🤖 Assistant: Your friend has developed a sense of excitement and enthusiasm the meeting is witnessing. They're both prepared to work on the same page, feeling a bit like they're changing the boundaries between their friendships and the conversation, and sharing their thoughts and request information can help break the gap between the conversation and the

'Your friend has developed a sense of excitement and enthusiasm the meeting is witnessing. They\'re both prepared to work on the same page, feeling a bit like they\'re changing the boundaries between their friendships and the conversation, and sharing their thoughts and request information can help break the gap between the conversation and the conversation.\n\nYour friend is poring over the conversation into a "vix," where the conversation is connected to the subject of the conversation and used to indicate the person is hungry or uncertain. It\'s possible that the conversation is re-taught and included in the conversation, such as in the chat.\n\nBefore the conversation, be sensitive and ask the question: How will you respond? Are there any specific concerns or conflicts being addressed within the conversation? If so, seek clarification on what\'s particularly important for the conversation.\n\nParagraphs such as:\n\n* Were there any significant changes or changes that could be made 

## Your Turn to Chat!

Now it's your turn to experiment with your model. Try different types of conversations:

- **Questions**: Ask about various topics
- **Creative tasks**: Ask for stories, poems, or creative writing
- **Problem solving**: Present problems and ask for solutions
- **Role-playing**: Have the model take on different personas
- **Multi-turn**: Build complex conversations with context

Use the functions below to chat with your model!


In [ ]:
# Single-turn chat - try your own messages!- pretty decent
chat_once("Explain what a training dataset and a test dataset are, and why we need both.")


👤 You: Explain what a training dataset and a test dataset are, and why we need both.
🤖 Assistant: A training dataset is a dataset that represents the amount of data you'll be working with. It's a collection of data points, each representing a specific number of distinct categories. Each dataset has its distinct characteristics, including the number of data points, the number of observations, and the context of the dataset.

In summary, training dataset is a useful tool for identifying patterns, patterns, and relationships between datasets, and it does not require additional testing or data visualization. It can also provide valuable insights into how you'll be working with datasets, such as tasks, data visualization, or customer demographics.

Additionally, training dataset is a data structure that provides a clear and concise way to communicate insights, insights, and insights. It can also be




"A training dataset is a dataset that represents the amount of data you'll be working with. It's a collection of data points, each representing a specific number of distinct categories. Each dataset has its distinct characteristics, including the number of data points, the number of observations, and the context of the dataset.\n\nIn summary, training dataset is a useful tool for identifying patterns, patterns, and relationships between datasets, and it does not require additional testing or data visualization. It can also provide valuable insights into how you'll be working with datasets, such as tasks, data visualization, or customer demographics.\n\nAdditionally, training dataset is a data structure that provides a clear and concise way to communicate insights, insights, and insights. It can also be"

In [ ]:
#pretty decent
chat_once("Explain gradient descent in simple terms.")

👤 You: Explain gradient descent in simple terms.
🤖 Assistant: Linear integration in simple terms can be used to predict a linear relationship. In this case, a slope of 0.05 and a slope of 0.05 suggests a non-linear relationship between variables. However, this relationship is not linear, as it does not have linear relationships between variables.

Linear integration can be used to predict a linear relationship between two continuous variables, which is often denoted by the fact that the values are all independent variables. The slope of 0.05 suggests that the slope of 0.05 implies a linear relationship between the variables. This suggests that a non-linear relationship between variables can be influenced by various factors, such as the degree of non-linearity or the number of variables in the distribution.




'Linear integration in simple terms can be used to predict a linear relationship. In this case, a slope of 0.05 and a slope of 0.05 suggests a non-linear relationship between variables. However, this relationship is not linear, as it does not have linear relationships between variables.\n\nLinear integration can be used to predict a linear relationship between two continuous variables, which is often denoted by the fact that the values are all independent variables. The slope of 0.05 suggests that the slope of 0.05 implies a linear relationship between the variables. This suggests that a non-linear relationship between variables can be influenced by various factors, such as the degree of non-linearity or the number of variables in the distribution.'

In [ ]:
# Single-turn chat - try your own messages! -not great
chat_once("Walk me step by step through how backpropagation works in a neural network.")

👤 You: Walk me step by step through how backpropagation works in a neural network.
🤖 Assistant: I can approach this problem by using a machine learning algorithm that is widely used for the control of neural networks in humans. The goal is to create a neural network that can be used to communicate neural networks in complex, dynamic ways, without relying on a single neural network.

The neural network is a network of neurons that can adapt and control a function, which can be used to manipulate and manipulate neural networks in the context of neural network dynamics. The neural networks are often used to manipulate and manipulate neural networks in response to changes in the neural network's behavior, such as changes in entry point frequencies, or changes in brain activity.

To further optimize the neural network, the neural network is also used to examine the neural network's behavior,




"I can approach this problem by using a machine learning algorithm that is widely used for the control of neural networks in humans. The goal is to create a neural network that can be used to communicate neural networks in complex, dynamic ways, without relying on a single neural network.\n\nThe neural network is a network of neurons that can adapt and control a function, which can be used to manipulate and manipulate neural networks in the context of neural network dynamics. The neural networks are often used to manipulate and manipulate neural networks in response to changes in the neural network's behavior, such as changes in entry point frequencies, or changes in brain activity.\n\nTo further optimize the neural network, the neural network is also used to examine the neural network's behavior,"

In [ ]:
# Single-turn chat - try your own messages!-on a tangent - not great
chat_once("Design a new AI assistant for a hospital and describe its capabilities in detail.")


👤 You: Design a new AI assistant for a hospital and describe its capabilities in detail.
🤖 Assistant: Imagine a medical assistant that is equipped with a sophisticated and safe environment that can withstand the stresses of the chaos. The assistant, on the other hand, is a skilled medical assistant that can withstand the stresses of the chaos. The assistant is trained to provide accurate medical records, complete with detailed medical records, medical records, and a detailed medical history.

The assistant's basic skills include:

Artificial intelligence and machine learning, which is a valuable tool in medical imaging and biotechnology. The assistant is trained to understand and respond to complex medical conditions.

Self-reflection and communication skills, which involve audio and nonverbal communication.

Transitioners and phone assistants, which can be used to communicate with patients and provide targeted




"Imagine a medical assistant that is equipped with a sophisticated and safe environment that can withstand the stresses of the chaos. The assistant, on the other hand, is a skilled medical assistant that can withstand the stresses of the chaos. The assistant is trained to provide accurate medical records, complete with detailed medical records, medical records, and a detailed medical history.\n\nThe assistant's basic skills include:\n\nArtificial intelligence and machine learning, which is a valuable tool in medical imaging and biotechnology. The assistant is trained to understand and respond to complex medical conditions.\n\nSelf-reflection and communication skills, which involve audio and nonverbal communication.\n\nTransitioners and phone assistants, which can be used to communicate with patients and provide targeted"

In [ ]:
# Multi-turn chat - start a conversation!

chat_session.clear_history()
chat_session.chat("Hey, I'm new to machine learning. Can you give me a quick overview?")


🗑️  Conversation history cleared!
👤 You: Hey, I'm new to machine learning. Can you give me a quick overview?
🤖 Assistant: Imagine you're walking into a city hall called Kaelbrundurna, a historic building that was constructed from the formation of MUSoa, Lu Quora and St. Thomas Cathedral. Founded in 1465, Kaelbrundurna, this structure is thought to have been built by Drosophital, a Greek architect of the 16th century that was part of our development. These design elements fell to the table, breaking apart once more and expanded, including the intricate varico-soothing curves, complex elements, and even geometric shapes.

Another landmark, Thomas Blackstone, is the cornerstone of our architecture, centered around the Agnes Museum of Indian Art. Blackstone is a sleek, 84-meter fireplace, with two towers standing close together, each with a colorful display of black and white lighting. Despite his team's support system, thebuilder's priorities remain focused on creating a park of landmarks

"Imagine you're walking into a city hall called Kaelbrundurna, a historic building that was constructed from the formation of MUSoa, Lu Quora and St. Thomas Cathedral. Founded in 1465, Kaelbrundurna, this structure is thought to have been built by Drosophital, a Greek architect of the 16th century that was part of our development. These design elements fell to the table, breaking apart once more and expanded, including the intricate varico-soothing curves, complex elements, and even geometric shapes.\n\nAnother landmark, Thomas Blackstone, is the cornerstone of our architecture, centered around the Agnes Museum of Indian Art. Blackstone is a sleek, 84-meter fireplace, with two towers standing close together, each with a colorful display of black and white lighting. Despite his team's support system, thebuilder's priorities remain focused on creating a park of landmarks, landmarks, and public transportation that would provide a unique"

In [ ]:
chat_session.chat("I think you've gone on some history lesson, I asked about machine learning")

👤 You: I think you've gone on some history lesson, I asked about machine learning
🤖 Assistant: You both think about building a good concrete foundation by building a beautiful, living public building. Growing up in Togo, Kaelbrundurna is known for its promise of climbing out and climbing as an architect. Thank you for your investment for your financial compensation.

Firstly, Kaelbrundurna is often the official home to a charismatic and warm-born entrepreneur namedAfter Schuet, as he is tasked with advancing generations of entrepreneurs through experimental projects and designing new designs. Kaelbrundurna has been exploring the world's largest rooftop building and is eager to spread its wealth and build its home. For now, Kaelbrundurna, where he is ready to venture into space, is struggling to adapt to changing conditions.

Secondly,Kaelbrundurna is also the Simil district building of the Meiji School, a city with a mystical atmosphere. The school has found itself struggling to adapt 

"You both think about building a good concrete foundation by building a beautiful, living public building. Growing up in Togo, Kaelbrundurna is known for its promise of climbing out and climbing as an architect. Thank you for your investment for your financial compensation.\n\nFirstly, Kaelbrundurna is often the official home to a charismatic and warm-born entrepreneur namedAfter Schuet, as he is tasked with advancing generations of entrepreneurs through experimental projects and designing new designs. Kaelbrundurna has been exploring the world's largest rooftop building and is eager to spread its wealth and build its home. For now, Kaelbrundurna, where he is ready to venture into space, is struggling to adapt to changing conditions.\n\nSecondly,Kaelbrundurna is also the Simil district building of the Meiji School, a city with a mystical atmosphere. The school has found itself struggling to adapt to changing circumstances"

## Tips for Better Conversations

### Parameter Tuning
- **Temperature (0.1 - 1.0)**: Lower = more focused, Higher = more creative
- **Max tokens (50 - 500)**: Controls response length

### Conversation Strategies
- **Be specific**: Clear questions get better answers
- **Provide context**: Multi-turn conversations work better with context
- **Experiment**: Try different conversation styles and topics
- **Iterate**: Adjust parameters based on the responses you get

### Common Issues
- **Short responses**: Increase `max_tokens`
- **Repetitive responses**: Adjust `temperature`
- **Off-topic responses**: Provide more context or be more specific

## Model Information

Your model was trained using:
- **Architecture**: GPT with RoPE positional encoding
- **Training**: Supervised Fine-Tuning on conversational data
- **Special tokens**: `<|user|>`, `<|assistant|>`, `<|end|>`
- **Context length**: 1024 tokens

Enjoy exploring your AI assistant! 🚀
